# Getrennte Modelle: RBF-SVM -> RBF-SVM

**Hypothese:** Zwei unabhängig trainierte, globale RBF-SVMs für Crop und Stage erreichen ohne Crop-bedingtes Routing mindestens die kombinierte Balanced Accuracy der hierarchischen RBF-SVM.

Crop und Stage werden getrennt vorhergesagt und erst danach zusammengefügt. Im Gegensatz zur kombinierten und hierarchischen Variante können dabei Paare entstehen, die im Training nicht vorkommen. Deren Anteil wird deshalb explizit ausgewertet.

Die Hyperparameter-Auswahl erfolgt ausschließlich auf den fünf Trainingsfolds. Der gemeinsame Validierungssplit wird erst für den CV-Gewinner einmal verwendet.

In [ ]:
%load_ext autoreload
%autoreload 2

from IPython.display import display

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_independent_svm
from awp2.preprocessing import PreprocessingConfig

PREPROCESSING = PreprocessingConfig(scale=True, use_meta=True)
C_VALUES = [30.0, 100.0, 300.0]
GAMMA_VALUES = [0.001, 0.003, 0.01]

## Hyperparameter suchen

Für Crop und Stage werden `C` und `gamma` unabhängig gewählt. Die 81 Kombinationen vergleichen damit nicht nur eine gemeinsame SVM-Konfiguration, sondern ob die beiden Targets unterschiedliche RBF-Skalen oder Regularisierung benötigen.

In [ ]:
tuned = tune(
    make_independent_svm(),
    TuneConfig(
        name="independent_svm_search",
        approach="independent",
        description=(
            "CV-only: Unabhängige RBF-SVMs für Crop und Stage; "
            "C und gamma je Ziel vergleichen."
        ),
        preprocessing=PREPROCESSING,
        param_grid={
            "crop_estimator__C": C_VALUES,
            "crop_estimator__gamma": GAMMA_VALUES,
            "stage_estimator__C": C_VALUES,
            "stage_estimator__gamma": GAMMA_VALUES,
        },
    ),
)

display(tuned.candidates.sort_values("rank"))
{
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## Einmalige Holdout-Bewertung

Diese Zelle wird genau einmal nach der CV-Entscheidung ausgeführt. Ihr Validierungsergebnis bestimmt keine weiteren Hyperparameter oder Paar-Korrekturen.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="independent_svm",
        approach="independent",
        description=(
            "Beste unabhängige Crop- und Stage-RBF-SVM aus independent_svm_search, "
            "einmal auf der gemeinsamen Validierung bewertet."
        ),
        preprocessing=PREPROCESSING,
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## Auswertung

Der Vergleich zeigt, ob die fehlende Fehlerfortpflanzung den Nachteil des globalen Stage-Modells ausgleicht. Der Anteil ungültiger Kombinationen bleibt Teil des Ergebnisses und wird nicht nachträglich korrigiert.

In [ ]:
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

HIERARCHICAL_SVM_CV_BACC_COMBINED = 0.839
HIERARCHICAL_SVM_VALIDATION_BACC_COMBINED = 0.844
COMBINED_SVM_VALIDATION_BACC_COMBINED = 0.866

comparison = {
    "CV BAcc kombiniert": {
        "Hierarchische RBF-SVM": HIERARCHICAL_SVM_CV_BACC_COMBINED,
        "Unabhängige RBF-SVMs": tuned.best_score,
    },
    "Validierung BAcc kombiniert": {
        "Kombinierte RBF-SVM": COMBINED_SVM_VALIDATION_BACC_COMBINED,
        "Hierarchische RBF-SVM": HIERARCHICAL_SVM_VALIDATION_BACC_COMBINED,
        "Unabhängige RBF-SVMs": result.metrics.bacc_combined,
    },
    "Ungültige Paare": result.metrics.invalid_combinations,
}
display(comparison)

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="crop_estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Getrennte Modelle: CV-Score der unabhängigen RBF-SVMs")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

{
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "metrics": result.metrics.model_dump(),
    "invalid_combinations": result.metrics.invalid_combinations,
}

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(tuning_ax.figure, "independent_svm_tuning", MODEL_DOCS_FIGURES_DIR)

confusion_figure = plot_confusion_matrices(result.y_val, result.y_pred)
save_doc_figure(confusion_figure, "independent_svm_confusion", MODEL_DOCS_FIGURES_DIR)

recall_figure = plot_class_recall(result.y_val, result.y_pred)
save_doc_figure(recall_figure, "independent_svm_recall", MODEL_DOCS_FIGURES_DIR)

confusions_figure = plot_top_confusions(result.y_val, result.y_pred)
save_doc_figure(confusions_figure, "independent_svm_confusions", MODEL_DOCS_FIGURES_DIR)

## Fazit

Die Entscheidung erfolgt anhand der CV-BAcc und der ungültigen Paare. Bei ähnlicher CV-Leistung zur hierarchischen SVM zeigt ein hoher Anteil ungültiger Paare, dass die Crop-Stage-Abhängigkeit für diese Aufgabe relevant ist.